# Assignment 1 — The Headline Test
**Muhammad Saad Noor | ERP ID: 37179**

> **Important:** This notebook runs on `teaching_prices_clean_SYNTHETIC.csv`, an invented practice dataset made for learning purposes because the original course CSV could not be obtained. Its outputs are demonstrations only and say nothing about real market prices.

## 1. Load the data
Place the CSV in the same folder as this notebook before running.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("teaching_prices_clean_SYNTHETIC.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()

## 2. Take a first look at the data

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values per column:\n", df.isna().sum())
print("\nUnique items:", df["item"].nunique())
print("Unique cities:", df["city"].nunique())
print("Unique dates:", df["date"].nunique())
print("\nData types:\n", df.dtypes)

## 3. Confirm the layout is what we expect

In [ ]:
# One row should exist for every date, city and item combination.
dupes = df.duplicated(subset=["date", "city", "item"]).sum()
print("Repeated date-city-item combinations:", dupes)

full_grid = df["date"].nunique() * df["city"].nunique() * df["item"].nunique()
print("Row count if the grid is complete:", full_grid)
print("Row count actually found:", len(df))

## 4. Question 1 — How did each item's price move across the weeks?

In [ ]:
weekly = df.groupby(["date", "item"], as_index=False)["price_pkr"].mean()

for name, grp in weekly.groupby("item"):
    plt.figure(figsize=(8, 4))
    plt.plot(grp["date"], grp["price_pkr"], marker="o")
    plt.title(f"Average recorded price by week: {name}")
    plt.xlabel("Observation date")
    plt.ylabel("Price (PKR per stated unit)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

change = (
    weekly.sort_values("date")
    .groupby("item")
    .agg(starting_price=("price_pkr", "first"),
         ending_price=("price_pkr", "last"))
)
change["absolute_change"] = change["ending_price"] - change["starting_price"]
change["percent_change"] = change["absolute_change"] / change["starting_price"] * 100
change.round(2)

## 5. Question 2 — Did prices differ between the two cities?

In [ ]:
avg_by_city = df.groupby(["item", "city"], as_index=False)["price_pkr"].mean()
city_table = avg_by_city.pivot(index="item", columns="city", values="price_pkr")
city_table["Islamabad_minus_Karachi"] = city_table["Islamabad"] - city_table["Karachi"]
city_table.round(2)

## 6. Match the cities date by date

In [ ]:
paired = df.pivot_table(
    index=["date", "item"], columns="city", values="price_pkr"
).reset_index()
paired["Islamabad_minus_Karachi"] = paired["Islamabad"] - paired["Karachi"]
paired.groupby("item")["Islamabad_minus_Karachi"].agg(["mean", "min", "max"]).round(2)

## 7. What we can and cannot conclude

### What this exercise can demonstrate
- How to check rows, columns, data types and missing values.
- How to compute averages and percentage changes.
- How to compare one item across cities and across dates.
- How plots make patterns easier to spot.

### What this exercise cannot demonstrate
- Real prices in Karachi or Islamabad.
- Whether food prices in Pakistan are actually rising.
- Whether either city is really more expensive in practice.
- Why any price changed.

Every number here comes from invented values. Do not present them as real observations or as market evidence.